# 데이터 준비 — NYC Yellow Taxi 결제수단(payment_type) 예측

Pandas/Polars 로딩 비교, 결측치·중복 처리, 기본 EDA를 수행하고
결제수단(3-클래스: 신용카드/현금/Flex Fare) 예측에 쓸 최종 학습 테이블을 만든다.

**컬럼 포함/제외 결정 근거**는 `README.md`(팀 EDA 인계서)와 본 노트북 내 자체 검증을 따른다.
각 결정의 상세 근거·표는 `data_preparation_report.md` 참고.

**데이터프레임 이름 규칙**: 각 단계에서 무엇을 했는지 알 수 있도록 `<처리내용>_df` 형식으로 이름을 붙인다.
`df_pandas` → `dedup_df`(중복 제거) → `column_dropped_df`(누수·저분산 컬럼 제외) → `outlier_filtered_df`(이상치·범위 필터링) → `feature_df`(최종 피처 테이블)

## 1. 라이브러리 Import 및 설정

In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import polars as pl

DATA_PATH = Path("yellow_tripdata_2026-05.parquet")
CLEANED_DATA_PATH = Path("payment_type_dataset.parquet")

PAYMENT_LABELS = {1: "신용카드", 2: "현금", 0: "Flex Fare"}
TARGET_CLASSES = list(PAYMENT_LABELS.keys())

# TLC taxi_zone_lookup 기준 공항 구역 (JFK=132, LaGuardia=138, Newark=1)
# PULocationID=132(JFK)는 다른 상위 픽업 구역과 결제수단 구성이 확연히 다름
# (평균요금 $62 vs $15~18, 평균거리 15mi vs 2~3mi, Flex Fare 0.21% vs 10~37%, 현금 27% vs 6~10%)
# -> 265개 구역 원-핫에 신호를 묻지 않고 is_airport로 직접 노출
AIRPORT_ZONE_IDS = {132, 138, 1}

# 실제 트립 특성만 피처로 사용 (제외 근거는 data_preparation_report.md 표 참고)
# cbd_congestion_fee, tolls_amount: 위치(구역) 기반 값이라 결제수단과 무관 -> 포함
NUMERIC_FEATURES = ["trip_distance", "fare_amount", "trip_duration", "cbd_congestion_fee", "tolls_amount"]
CATEGORICAL_FEATURES = ["PULocationID", "DOLocationID", "hour", "day_of_week", "VendorID", "is_airport"]
TARGET_COLUMN = "payment_label"

# 결측치가 payment_type=0(Flex Fare)과 100% 일치해 피처로 쓰면 누수가 되는 컬럼
LEAKAGE_MISSING_COLUMNS = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee",
]
# 정답(payment_type)과 구조적으로 얽혀 누수가 되는 컬럼
# extra: Flex Fare는 99.0%가 0인 반면 카드 44.5%/현금 48.0%만 0 -> tip_amount와 같은 맥락의 준-누수로 판단해 제외
LEAKAGE_TARGET_COLUMNS = ["tip_amount", "total_amount", "extra"]
# 값이 거의 상수라 결제수단 구분에 정보량이 없는 컬럼 (README.md 범주형 표)
LOW_VARIANCE_COLUMNS = ["mta_tax", "improvement_surcharge"]

# 파생 컬럼을 만드는 데만 쓰이고 최종 테이블에는 들어가지 않는 원본 컬럼
DERIVED_SOURCE_COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "payment_type"]

VALID_PICKUP_START = pd.Timestamp("2026-05-01")
VALID_PICKUP_END = pd.Timestamp("2026-06-01")

print("설정 완료")

설정 완료


## 2. Pandas vs Polars 로딩 비교

같은 parquet 파일을 두 라이브러리로 각각 로딩해 shape과 소요시간을 비교한다.
(로딩 이후 단계는 pandas로 일관되게 진행 — 이유는 data_preparation_report.md "Pandas vs Polars 사용 범위" 참고)

In [2]:
start = perf_counter()
df_pandas = pd.read_parquet(DATA_PATH)
pandas_seconds = perf_counter() - start

start = perf_counter()
df_polars = pl.read_parquet(DATA_PATH)
polars_seconds = perf_counter() - start

print(f"Pandas : shape={df_pandas.shape}, 소요시간={pandas_seconds:.2f}초")
print(f"Polars : shape={df_polars.shape}, 소요시간={polars_seconds:.2f}초")

Pandas : shape=(4090836, 20), 소요시간=0.22초
Polars : shape=(4090836, 20), 소요시간=0.10초


## 3. 기본 EDA — 구조 확인

행/열 개수, dtype 구성을 확인한다.

In [3]:
print(f"행 수: {len(df_pandas):,}, 컬럼 수: {df_pandas.shape[1]}")
print()
print("dtype 구성:")
print(df_pandas.dtypes.value_counts())

행 수: 4,090,836, 컬럼 수: 20

dtype 구성:
float64           13
int32              3
datetime64[us]     2
str                1
int64              1
Name: count, dtype: int64


## 4. 중복 처리

완전 중복 행을 확인하고 제거한다. **컬럼을 줄이기 전, 원본(20개 컬럼) 상태에서 먼저 검사한다** —
컬럼을 먼저 지우고 나면 서로 다른 트립이 남은 컬럼값만 우연히 같아서 잘못 "중복"으로 잡힐 수 있기 때문.

In [4]:
before = len(df_pandas)
dedup_df = df_pandas.drop_duplicates()
removed = before - len(dedup_df)

print(f"중복 제거: {removed:,}건 제거 ({before:,} -> {len(dedup_df):,})")

중복 제거: 0건 제거 (4,090,836 -> 4,090,836)


## 5. 결측치 확인 및 누수·저분산 컬럼 제외

결측치가 있는 5개 컬럼은 전부 `payment_type=0`(Flex Fare)과 100% 일치해 피처로 쓰면 타깃이 노출된다.
`tip_amount`·`total_amount`·`extra`도 같은 맥락의 누수, `mta_tax`·`improvement_surcharge`는 저분산이라 정보량이 없다.
확인과 동시에 바로 제외한다 (자세한 근거는 `data_preparation_report.md`의 "삭제한 컬럼" 표 참고).

In [5]:
missing = dedup_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"결측치가 있는 컬럼: {list(missing.index)} (전부 {missing.iloc[0]:,}건, payment_type=0과 100% 일치)")

columns_to_drop = LEAKAGE_MISSING_COLUMNS + LEAKAGE_TARGET_COLUMNS + LOW_VARIANCE_COLUMNS
column_dropped_df = dedup_df.drop(columns=columns_to_drop)

print(f"\n제외 컬럼 ({len(columns_to_drop)}개): {columns_to_drop}")
print(f"컬럼 제외 후 shape: {column_dropped_df.shape}")

결측치가 있는 컬럼: ['passenger_count', 'RatecodeID', 'store_and_fwd_flag', 'congestion_surcharge', 'Airport_fee'] (전부 955,371건, payment_type=0과 100% 일치)

제외 컬럼 (10개): ['passenger_count', 'RatecodeID', 'store_and_fwd_flag', 'congestion_surcharge', 'Airport_fee', 'tip_amount', 'total_amount', 'extra', 'mta_tax', 'improvement_surcharge']
컬럼 제외 후 shape: (4090836, 10)


## 6. 이상치·범위 필터링

근거가 확인된 규칙대로 행을 순서대로 제거하고 파생변수를 생성한다.

- `fare_amount > 0`
- `trip_distance > 0` — 0마일인데 평균 \$30 요금이 부과됨 → 미터기/GPS 오류로 판단 (README.md 잠재 결측 표)
- `trip_distance < 200` — 레버리지 효과를 유발하는 극단값 제거 (자체 검증: 이 필터 없이는 trip_distance↔fare_amount 상관계수가 0.008로 왜곡됨, `<200` 적용 시 0.83대로 안정화)
- `~(fare_amount >= 100 & trip_distance < 1)` — 요금-거리 불일치 제거 (자체 검증: 이 549건은 대부분 `RatecodeID=5`(협상요금)·거리<1마일인데 요금이 \$100 이상 찍힌 행으로, 미터기/GPS 기록 오류로 판단. 제거 시 trip_distance↔fare_amount 상관계수가 0.8623→0.8675로 오히려 개선되어 정상 신호가 아님을 뒷받침함)
- `payment_type in {0,1,2}` — 3-클래스 타깃만 유지
- `tpep_pickup_datetime`이 2026-05 범위 내
- `0 < trip_duration < 180(분)`

> **`fare_amount`에는 상한(cap)을 두지 않는다.** `trip_distance`와 달리 상한을 $150~1000 어디로 바꿔도
> trip_distance↔fare_amount 상관계수가 0.862~0.868 범위에서 거의 안 움직이고, StandardScaler 표준화 후
> 정상권(p10~p90) 값의 퍼짐도 2.02~2.13(z-score 폭)으로 5% 안팎만 차이 나 스케일링 왜곡 근거도 약하다.
> 반면 `RatecodeID`가 4(Nassau/Westchester)·5(협상요금)인 실제 장거리 정상 운행은 요금이 \$96(중앙값)~\$881(최댓값)까지
> 자연스럽게 분포해서, \$150~250 같은 낮은 상한을 걸면 이 정상 운행의 최대 18%가 잘려나간다.
> `trip_distance`가 문제였던 건 원본 최댓값이 정상권(99%ile)의 15,793배였기 때문인데, `fare_amount` 최댓값(\$1,061.72)은
> 정상 장거리 99%ile(\$350)의 3배 수준이라 같은 논리가 적용되지 않는다. 자세한 수치는 `data_preparation_report.md` 참고.

In [6]:
working_df = column_dropped_df
steps = []

def apply_filter(mask, label):
    global working_df
    before = len(working_df)
    working_df = working_df[mask]
    steps.append((label, before, len(working_df)))

apply_filter(working_df["fare_amount"] > 0, "fare_amount > 0")
apply_filter(
    working_df["trip_distance"] > 0,
    "trip_distance > 0 (0마일인데 평균 $30 요금 부과 -> 미터기/GPS 오류로 판단)",
)
apply_filter(
    working_df["trip_distance"] < 200,
    "trip_distance < 200 (레버리지 효과 유발 극단값 제거, 자체 검증)",
)
apply_filter(
    ~((working_df["fare_amount"] >= 100) & (working_df["trip_distance"] < 1)),
    "fare_amount>=100 & trip_distance<1 제거 (요금-거리 불일치, 미터기/GPS 오류로 판단, 자체 검증)",
)
apply_filter(
    working_df["payment_type"].isin(TARGET_CLASSES),
    "payment_type in {0,1,2} (3-클래스 타깃만 유지)",
)
apply_filter(
    (working_df["tpep_pickup_datetime"] >= VALID_PICKUP_START)
    & (working_df["tpep_pickup_datetime"] < VALID_PICKUP_END),
    "tpep_pickup_datetime이 2026-05 범위 내",
)

working_df = working_df.copy()
working_df["trip_duration"] = (
    working_df["tpep_dropoff_datetime"] - working_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60
apply_filter(
    (working_df["trip_duration"] > 0) & (working_df["trip_duration"] < 180),
    "0 < trip_duration < 180(분)",
)

for label, before, after in steps:
    print(f"- {label}: {before:,} -> {after:,}건 ({before - after:,}건 제거)")

outlier_filtered_df = working_df

- fare_amount > 0: 4,090,836 -> 4,073,654건 (17,182건 제거)
- trip_distance > 0 (0마일인데 평균 $30 요금 부과 -> 미터기/GPS 오류로 판단): 4,073,654 -> 3,962,811건 (110,843건 제거)
- trip_distance < 200 (레버리지 효과 유발 극단값 제거, 자체 검증): 3,962,811 -> 3,962,738건 (73건 제거)
- fare_amount>=100 & trip_distance<1 제거 (요금-거리 불일치, 미터기/GPS 오류로 판단, 자체 검증): 3,962,738 -> 3,962,189건 (549건 제거)
- payment_type in {0,1,2} (3-클래스 타깃만 유지): 3,962,189 -> 3,942,906건 (19,283건 제거)
- tpep_pickup_datetime이 2026-05 범위 내: 3,942,906 -> 3,942,892건 (14건 제거)
- 0 < trip_duration < 180(분): 3,942,892 -> 3,891,255건 (51,637건 제거)


In [7]:
# 원본 datetime은 값이 거의 다 달라 모델이 못 배우니, 반복되는 시간/요일 패턴만 추출
outlier_filtered_df["hour"] = outlier_filtered_df["tpep_pickup_datetime"].dt.hour
outlier_filtered_df["day_of_week"] = outlier_filtered_df["tpep_pickup_datetime"].dt.dayofweek
outlier_filtered_df["payment_label"] = outlier_filtered_df["payment_type"].map(PAYMENT_LABELS)

# 공항(JFK/LaGuardia/Newark) 픽업 또는 하차 여부. PULocationID/DOLocationID를
# 265개 구역 그대로 원-핫하면 공항 신호가 희소 행렬 속에 묻혀 해석이 어려워짐 -> 이진 플래그로 명시
outlier_filtered_df["is_airport"] = (
    outlier_filtered_df["PULocationID"].isin(AIRPORT_ZONE_IDS)
    | outlier_filtered_df["DOLocationID"].isin(AIRPORT_ZONE_IDS)
).astype(int)

print(f"필터링 후 shape: {outlier_filtered_df.shape}")
print(f"is_airport=1 비율: {outlier_filtered_df['is_airport'].mean()*100:.2f}%")

필터링 후 shape: (3891255, 15)
is_airport=1 비율: 8.01%


## 7. 최종 피처 테이블 구성

사용 컬럼만 선택해 최종 학습용 피처 테이블을 만든다. 원본에 검토되지 않은 새 컬럼이 섞여 있으면
조용히 빠지지 않고 경고가 뜨도록 안전장치를 둔다 (나중에 데이터가 바뀌어도 누수 컬럼을 놓치지 않기 위함).

In [8]:
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]

reviewed_columns = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES + columns_to_drop + DERIVED_SOURCE_COLUMNS)
derived_only_columns = {"hour", "day_of_week", "trip_duration", "payment_label"}
unreviewed = set(outlier_filtered_df.columns) - reviewed_columns - derived_only_columns
if unreviewed:
    print(f"경고: 검토되지 않은 컬럼 발견: {unreviewed} -- 포함 여부 검토 후 반영 필요")
else:
    print("검토되지 않은 컬럼 없음")

feature_df = outlier_filtered_df[FEATURE_COLUMNS].dropna()
print(f"\n사용 컬럼 ({len(FEATURE_COLUMNS)}개): {FEATURE_COLUMNS}")
print(f"최종 shape: {feature_df.shape}")

검토되지 않은 컬럼 없음

사용 컬럼 (12개): ['trip_distance', 'fare_amount', 'trip_duration', 'cbd_congestion_fee', 'tolls_amount', 'PULocationID', 'DOLocationID', 'hour', 'day_of_week', 'VendorID', 'is_airport', 'payment_label']
최종 shape: (3891255, 12)


In [9]:
counts = feature_df[TARGET_COLUMN].value_counts()
ratios = feature_df[TARGET_COLUMN].value_counts(normalize=True) * 100
pd.DataFrame({"count": counts, "ratio(%)": ratios.round(2)})

,count,ratio(%)
payment_label,,
신용카드,2660128,68.36
Flex Fare,878188,22.57
현금,352939,9.07


## 8. 결과 저장

다음 단계(시각화·통계분석·ML Pipeline)가 공통으로 사용할 수 있도록 정제된 테이블을 저장한다.

In [10]:
feature_df.to_parquet(CLEANED_DATA_PATH, index=False)
print(f"정제된 데이터 저장 완료: {CLEANED_DATA_PATH}")

정제된 데이터 저장 완료: payment_type_dataset.parquet
